## Here we stack maps in order to fill the holes due to the scanning strategy before fitting the order 0 of the Moon that will give us the line of sight of each detector

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2

In [ ]:
%config InlineBackend.figure_format='retina'
from IPython import display
display.display(display.HTML("<style>.container { width:95% !important; }</style>"))

# to have interactive plots : pip install ipympl
%matplotlib ipympl

### General imports
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pickle

plt.rc('figure',figsize=(10,10))
plt.rc('font',size=16)

### Astropy configuration
from astropy.stats import sigma_clip

import pipeline_moon_functions as pmf

In [ ]:
ang_size = 40 # degrees
xsize = 201
reso = ang_size*60/xsize # arcmin/pix
min_plot = -5e3
max_plot = 1.2e4
zenith = [0, 90]

In [ ]:
datadir_MoonProject = "/Users/huchet/Documents/code/scripts/MoonProject/"

In [ ]:
list_of_observations = [                  "2026-07-23_rise",
                        "2026-07-24_set", "2026-07-24_rise",
                        "2026-07-25_set", "2026-07-25_rise",
                        "2026-07-26_set", "2026-07-26_rise",
                        "2026-07-27_set", "2026-07-27_rise",
                        "2026-07-28_set"]

dict_already_computed_tshifts = {                         "2026-07-23_rise": -0.63,
                                 "2026-07-24_set": -0.65, "2026-07-24_rise": -0.63,
                                 "2026-07-25_set": -0.7,  "2026-07-25_rise": -0.6,
                                 "2026-07-26_set": -0.52, "2026-07-26_rise": -0.7,
                                 "2026-07-27_set": -0.65, "2026-07-27_rise": -0.85,
                                 "2026-07-28_set": -0.75}
n_obs = len(list_of_observations)
det_los = True # if we want to stack the maps with each detector line of sight at zenith

In [ ]:
# it is too heavy to load all full maps so we choose a submap centered on zenith with a given radius
center = zenith
radius = 30 # degrees
nside = 512
vec_centre = hp.ang2vec(center[0], center[1], lonlat=True)
chosen_patch = hp.query_disc(nside, vec_centre, np.radians(radius))
n_pix = len(chosen_patch)
n_TES = 256
# in order to go from patch to full map
remap = pmf.get_remapping_operator(qubic_patch=chosen_patch, shape_patch=np.shape(chosen_patch), shape_full=hp.nside2npix(nside))
    

# the TES for display, but the computation is done for all
TESNum = 218 #96
i_TES = TESNum - 1
allmaps_ = np.zeros((n_obs, n_TES, n_pix))

det_zen = ""
if det_los:
    det_zen += "_det"

for i_obs, obsname in enumerate(list_of_observations):
    ObsDate, scanning = obsname.split("_")
    tshift = dict_already_computed_tshifts[obsname]
    file_name = "allmaps-{}_{}_nside{}_zenith{}_tshift{}.pkl".format(ObsDate, scanning, nside, det_zen, tshift)
    allTESNum, allmaps = pickle.load( open( datadir_MoonProject + file_name, "rb" ) )
    allmaps_[i_obs] = allmaps[:, chosen_patch]
    print("Read {}".format(file_name))

In [ ]:
print(np.shape(allmaps_))

In [ ]:
allmaps_masked = np.ma.masked_array(allmaps_, allmaps_ == hp.UNSEEN)
stacked_map = np.mean(allmaps_masked, axis=0)
stacked_map[stacked_map.mask] = 0

In [ ]:
print(np.shape(allmaps_))
print(np.shape(stacked_map))
print(remap)
print(np.shape(allmaps_[i_obs, i_TES]))
print(np.shape(stacked_map[i_TES]))
print(np.shape(remap(allmaps_[i_obs, i_TES])))
print(np.shape(remap(stacked_map[i_TES])))

In [ ]:
for i_obs, obsname in enumerate(list_of_observations):
    hp.gnomview(remap(allmaps_[i_obs, i_TES]), reso=reso, min=min_plot, max=max_plot, xsize=xsize,
                        rot=center,return_projected_map=False, no_plot=False)
    plt.show()

In [ ]:
mapsb_proj = hp.gnomview(remap(stacked_map[i_TES]), reso=reso, min=min_plot, max=max_plot, xsize=xsize,
                        rot=center, return_projected_map=True, no_plot=True)
mapsb_proj[mapsb_proj > 1e10] = 0

In [ ]:
print(np.shape(allmaps_))

In [ ]:
fig, ax = plt.subplots()
ax.imshow(mapsb_proj, vmin=min_plot, vmax=max_plot)
plt.show()

In [ ]:
allmaps_masked_ = sigma_clip(allmaps_masked, sigma=2, axis=0)
stacked_map_ = np.mean(allmaps_masked_, axis=0)
stacked_map_[stacked_map_.mask] = 0

In [ ]:
print(np.shape(allmaps_masked_.mask))

In [ ]:
# for i in range(len(list_of_observations)):
i = 0
hp.gnomview(remap(allmaps_masked_.mask[i, i_TES]), reso=reso, xsize=xsize,
                    rot=center,return_projected_map=False, no_plot=False)
plt.show()
hp.gnomview(remap((allmaps_ == hp.UNSEEN)[i, i_TES]), reso=reso, xsize=xsize,
                    rot=center,return_projected_map=False, no_plot=False)
plt.show()
hp.gnomview(remap(np.logical_and(~(allmaps_ == hp.UNSEEN)[i, i_TES], allmaps_masked_.mask[i, i_TES])), reso=reso, xsize=xsize,
                    rot=center,return_projected_map=False, no_plot=False)

In [ ]:
mapsb_proj_ = hp.gnomview(remap(stacked_map_[i_TES]), reso=reso, min=min_plot, max=max_plot, xsize=xsize,
                        rot=center, return_projected_map=True, no_plot=True)
mapsb_proj_[mapsb_proj_ == 1e20] = 0

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(mapsb_proj, vmin=min_plot, vmax=max_plot)
axs[1].imshow(mapsb_proj_, vmin=min_plot, vmax=max_plot)
axs[2].imshow(mapsb_proj_ - mapsb_proj, vmin=min_plot, vmax=max_plot)
plt.show()

In [ ]:
sigma_ok = np.std(allmaps_, where=allmaps_ != hp.UNSEEN, axis=0)
# sigma_ok[sigma_ok.mask] = 0

In [ ]:
sigma_proj = hp.gnomview(remap(sigma_ok[i_TES]), reso=reso, min=min_plot, max=max_plot, xsize=xsize,
                        rot=center, return_projected_map=True, no_plot=True)
fig, ax = plt.subplots()
ax.imshow(sigma_proj, vmin=min_plot, vmax=max_plot)
plt.show()

In [ ]:
# %%script echo skipping figure creation
# we plot the clipped maps and save them

more_name = "202607_stack"
title = "Stacked maps (July 2026)"
if det_los:
    more_name += "_detzenith"
    title += "det l.o.s. at zenith"
center = [0, 90]
xsize = 200
reso = 10
vmin = -5e3
vmax = 2.5e4
n0 = 8
figsize=[n0*3]*2
subtitle = "{}, {}x{} {}' pixels. colorbar from {:.2E} to {:.2E} ADU".format(title, xsize, xsize, reso, vmin, vmax)
# plt.figure(figsize=figsize)
fig, axs = plt.subplots(n0, n0, figsize=figsize, sharex=True, sharey=True, constrained_layout=True)
plt.suptitle(subtitle)
for i in range(len(allTESNum)):
    remap_stacked_map = remap(stacked_map_[i, :]).data
    ok = remap_stacked_map != hp.UNSEEN
    mm = np.zeros(12*nside**2) + hp.UNSEEN
    mm[ok] = remap_stacked_map[ok]
    img_proj = hp.gnomview(mm, reso=reso, xsize=xsize, min=vmin, max=vmax, 
                title=allTESNum[i], rot=center, return_projected_map=True, no_plot=True)
    img_masked = np.ma.masked_array(img_proj.data, img_proj.mask)
    i_plot = i - i//(n0**2) * n0**2 # for each plot we want i_plot to go from 0 to n0**2 - 1
    ax = axs[(i_plot//n0), (i_plot%n0)]
    ax.set_title(allTESNum[i])
    ax.set_facecolor("gray")
    ax.imshow(img_masked, vmin=vmin, vmax=vmax, origin="upper") # "upper" is the default, it puts the highest observed elevation at the top of the image
    if ((i + 1)%(n0**2))==0:
        figname = "figures/allTES{}_{}.pdf".format(more_name, (i + 1)//(n0**2))
        # plt.tight_layout(rect=[0, 0.05, 1, 0.90])
        plt.savefig(figname, dpi=150)
        print("Saved figure {}".format(figname))
        plt.show()
        if i < len(allTESNum) - 1:
            fig, axs = plt.subplots(n0, n0, figsize=figsize, sharex=True, sharey=True, constrained_layout=True)
            plt.suptitle(subtitle)

In [ ]:
# We save the clipped maps!
file_name = "allmaps-{}_nside{}_zenith.pkl".format(more_name, nside)

pickle.dump( [allTESNum, stacked_map_, chosen_patch], open( datadir_MoonProject + file_name, "wb" ) )
print("Saved {}".format(file_name))
# sys.exit()